# Malos Controles en Python
## Econometría Avanzada — Ana María Díaz

**Regla de oro:** controla variables que abren caminos espurios,  
pero **nunca** variables que están *en* el camino causal o que son efectos
tanto del tratamiento como del resultado.

| Tipo | Estructura | ¿Controlar? | Consecuencia |
|------|-----------|-------------|-------------|
| Mediador | D → M → Y | ❌ (para efecto total) | Bloquea el canal causal |
| Colisionador | D → C ← Y | ❌ nunca | Abre camino espurio |
| Proxy contaminado | D → P ← U → Y | ❌ | Atenúa el efecto real |

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.random.seed(2468)

---
## CASO 1: El Mediador — controlar bloquea el efecto que quieres medir

**Estructura causal:** Educación → Tipo de empleo → Salario

**Pregunta:** ¿Cuánto sube el salario por tener título universitario?

**Intuición:** La educación mejora el salario *porque* permite acceder a mejores empleos.  
Si controlas el tipo de empleo, le "preguntas" a la regresión:  
*"manteniendo el tipo de empleo fijo, ¿qué hace la educación?"*  
La respuesta es casi nada — porque ese era el único canal.

**¿Por qué parece buena idea?** Quizás el tipo de empleo tiene un efecto independiente,
y quieres "limpiar" ese canal. Pero si la educación actúa *solo a través* del empleo,
al controlar el mediador eliminas el efecto que querías medir.

In [ ]:
np.random.seed(2468)
n = 10000

# Parámetros del proceso generador de datos
efecto_educ_empleo    = 2  # La educación mejora el tipo de empleo
efecto_empleo_salario = 1  # El tipo de empleo sube el salario
# Efecto TOTAL de educación en salario = 2 × 1 = 2

educacion   = np.random.binomial(1, 0.5, n)                           # 1 = título universitario
tipo_empleo = efecto_educ_empleo * educacion + np.random.normal(0, 1, n)       # Mediador
salario     = efecto_empleo_salario * tipo_empleo + np.random.normal(0, 1, n)  # Resultado

print("=== CASO 1: MEDIADOR ===")
print("Efecto real de educacion en salario = 2\n")

# Regresión CORRECTA: solo educacion → salario (efecto TOTAL)
print("Regresión CORRECTA — sin controlar el mediador (efecto TOTAL):")
X_total = sm.add_constant(educacion)
modelo_total = sm.OLS(salario, X_total).fit()
print(f"  Coef. educacion: {modelo_total.params[1]:.4f}  (esperado ~2)")
print(f"  p-valor:         {modelo_total.pvalues[1]:.4f}\n")

# Regresión con MAL CONTROL: incluye tipo_empleo (mediador)
print("Regresión con MAL CONTROL — controlando el tipo de empleo:")
X_directo = sm.add_constant(np.column_stack([educacion, tipo_empleo]))
modelo_directo = sm.OLS(salario, X_directo).fit()
print(f"  Coef. educacion: {modelo_directo.params[1]:.4f}  (esperado ~0, ¡efecto desaparece!)")
print(f"  p-valor:         {modelo_directo.pvalues[1]:.4f}")
print("\n→ La educación sube el salario SOLO A TRAVÉS del tipo de empleo.")
print("  Al controlar el mediador, bloqueamos ese canal y el efecto desaparece.")

---
## CASO 2: El Colisionador — controlar abre un camino espurio

**Estructura causal:** Conexiones → Contratado ← Productividad  
(D y Y son independientes; ambas *causan* C)

**Pregunta:** ¿Tienen conexiones y productividad alguna relación?

**Intuición:** En la población general, conexiones y productividad no tienen nada que ver.  
Pero si miras solo a los **contratados** (o controlas por serlo), aparece una correlación negativa falsa:  
*"entre quienes lograron entrar, los que tenían más palancas necesitaban menos mérito."*  
Esa asociación es un artefacto estadístico, no una relación causal real.

**Nota:** Selección muestral es colisionador. Si estudias solo empleados, sobrevivientes,
o cualquier grupo que resulte de causas múltiples, puedes estar cayendo en este problema.

In [ ]:
np.random.seed(2468)
n = 5000

conexiones    = np.random.normal(0, 1, n)                                      # Palancas / red de contactos
productividad = np.random.normal(0, 1, n)                                      # Independiente de conexiones
contratado    = 2 * conexiones + 3 * productividad + np.random.normal(0, 1, n) # Colisionador

print("=== CASO 2: COLISIONADOR ===")
print("Efecto real de conexiones en productividad = 0 (son independientes)\n")

# Regresión CORRECTA
print("Regresión CORRECTA — sin controlar el colisionador:")
X_noC = sm.add_constant(conexiones)
modelo_noC = sm.OLS(productividad, X_noC).fit()
print(f"  Coef. conexiones: {modelo_noC.params[1]:.4f}  (esperado ~0)")
print(f"  p-valor:          {modelo_noC.pvalues[1]:.4f}\n")

# Regresión con MAL CONTROL
print("Regresión con MAL CONTROL — controlando ser contratado:")
X_conC = sm.add_constant(np.column_stack([conexiones, contratado]))
modelo_conC = sm.OLS(productividad, X_conC).fit()
print(f"  Coef. conexiones: {modelo_conC.params[1]:.4f}  (¡SESGADO! debería ser ~0)")
print(f"  p-valor:          {modelo_conC.pvalues[1]:.4f}")
print("\n→ Al condicionar en 'contratado', abrimos información espuria:")
print("  entre los contratados, más conexiones implican menos mérito propio.")

---
## CASO 3: El Proxy Contaminado — el "control" mezcla confusor y efecto

**Estructura causal:**
- Habilidad innata U (no observable) → Salario Y  ← Tratamiento D  
- Habilidad innata U → Test tardío P ← Tratamiento D

**Pregunta:** ¿Cuánto sube el salario el programa de empleabilidad?

**Intuición:** U confunde la estimación (personas con más habilidad entran más al programa  
Y ganan más). Queremos controlar U, pero no lo observamos.  
Usamos un test de habilidades aplicado **después del programa** como proxy.  
El problema: ese test también captura el efecto del programa.  
Al controlarlo, absorbemos parte del efecto D que queríamos medir.

**¿Por qué parece buena idea?** El test es una medida de habilidad, y habilidad confunde.
Parece razonable controlar. Pero el timing importa: si el proxy es posterior al tratamiento,
puede estar contaminado por el propio tratamiento.

In [ ]:
np.random.seed(2468)
n = 5000

habilidad_innata = np.random.normal(0, 1, n)                                          # U: no observable
tratamiento      = np.random.binomial(1, 0.5, n)                                      # D: asignado aleatoriamente
salario          = 2 * tratamiento + 1.5 * habilidad_innata + np.random.normal(0,1,n) # Y: efecto real = 2
test_tardio      = 0.5 * tratamiento + habilidad_innata + np.random.normal(0,0.5,n)   # P: proxy contaminado

print("=== CASO 3: PROXY CONTAMINADO ===")
print("Efecto real del tratamiento en salario = 2\n")

# Regresión CORRECTA (experimento aleatorio, no se necesita controlar U)
print("Regresión CORRECTA — sin controlar (RCT, D es aleatorio):")
X_sin = sm.add_constant(tratamiento)
modelo_sin = sm.OLS(salario, X_sin).fit()
print(f"  Coef. tratamiento: {modelo_sin.params[1]:.4f}  (esperado ~2)")
print(f"  p-valor:           {modelo_sin.pvalues[1]:.4f}\n")

# Regresión con PROXY CONTAMINADO
print("Regresión con PROXY CONTAMINADO — controlando el test tardío:")
X_proxy = sm.add_constant(np.column_stack([tratamiento, test_tardio]))
modelo_proxy = sm.OLS(salario, X_proxy).fit()
print(f"  Coef. tratamiento: {modelo_proxy.params[1]:.4f}  (atenuado, < 2)")
print(f"  p-valor:           {modelo_proxy.pvalues[1]:.4f}")
print("  → El proxy absorbe parte del efecto del tratamiento.\n")

# Regresión IDEAL (si pudiéramos observar U)
print("Regresión IDEAL — controlando la habilidad innata directamente:")
X_ideal = sm.add_constant(np.column_stack([tratamiento, habilidad_innata]))
modelo_ideal = sm.OLS(salario, X_ideal).fit()
print(f"  Coef. tratamiento: {modelo_ideal.params[1]:.4f}  (correcto, ~2)")
print(f"  p-valor:           {modelo_ideal.pvalues[1]:.4f}")
print("  → Controlar U directamente no distorsiona el efecto de D.")

---
## Monte Carlo — Distribución de coeficientes en los tres casos

500 repeticiones para ver que el sesgo no es accidental sino sistemático.

In [ ]:
n_reps = 500

# Almacenamiento
b_total   = np.zeros(n_reps); b_directo = np.zeros(n_reps)
b_sinC    = np.zeros(n_reps); b_conC    = np.zeros(n_reps)
b_sinP    = np.zeros(n_reps); b_proxy   = np.zeros(n_reps); b_ideal = np.zeros(n_reps)

for i in range(n_reps):

    # --- Caso 1: Mediador ---
    educ   = np.random.binomial(1, 0.5, 3000)
    empleo = 2 * educ + np.random.normal(0, 1, 3000)
    sal    = 1 * empleo + np.random.normal(0, 1, 3000)
    b_total[i]   = sm.OLS(sal, sm.add_constant(educ)).fit().params[1]
    b_directo[i] = sm.OLS(sal, sm.add_constant(np.column_stack([educ, empleo]))).fit().params[1]

    # --- Caso 2: Colisionador ---
    con  = np.random.normal(0, 1, 1000)
    prod = np.random.normal(0, 1, 1000)
    cont = 2 * con + 3 * prod + np.random.normal(0, 1, 1000)
    b_sinC[i] = sm.OLS(prod, sm.add_constant(con)).fit().params[1]
    b_conC[i] = sm.OLS(prod, sm.add_constant(np.column_stack([con, cont]))).fit().params[1]

    # --- Caso 3: Proxy contaminado ---
    U = np.random.normal(0, 1, 2000)
    D = np.random.binomial(1, 0.5, 2000)
    Y = 2 * D + 1.5 * U + np.random.normal(0, 1, 2000)
    P = 0.5 * D + U + np.random.normal(0, 0.5, 2000)
    b_sinP[i]  = sm.OLS(Y, sm.add_constant(D)).fit().params[1]
    b_proxy[i] = sm.OLS(Y, sm.add_constant(np.column_stack([D, P]))).fit().params[1]
    b_ideal[i] = sm.OLS(Y, sm.add_constant(np.column_stack([D, U]))).fit().params[1]

print(f"Mediador    — b_total: {b_total.mean():.3f} (esperado ~2) | b_directo: {b_directo.mean():.3f} (esperado ~0)")
print(f"Colisionador — b_sinC: {b_sinC.mean():.3f} (esperado ~0) | b_conC: {b_conC.mean():.3f} (sesgado)")
print(f"Proxy       — b_sinP: {b_sinP.mean():.3f} (~2) | b_proxy: {b_proxy.mean():.3f} (atenuado) | b_ideal: {b_ideal.mean():.3f} (~2)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Distribución de coeficientes — Monte Carlo (500 reps)", fontsize=13, fontweight='bold')

# Caso 1: Mediador
axes[0].hist(b_total,   bins=30, alpha=0.55, color='royalblue',  label=f'Correcto (~2): {b_total.mean():.2f}')
axes[0].hist(b_directo, bins=30, alpha=0.55, color='tomato',     label=f'Mal control (~0): {b_directo.mean():.2f}')
axes[0].axvline(2, color='royalblue', linestyle='--', lw=1.5)
axes[0].axvline(0, color='tomato',    linestyle='--', lw=1.5)
axes[0].set_title('Caso 1: Mediador\nCoef. de educacion')
axes[0].set_xlabel('Coeficiente estimado')
axes[0].legend(fontsize=8)

# Caso 2: Colisionador
axes[1].hist(b_sinC, bins=30, alpha=0.55, color='mediumseagreen', label=f'Correcto (~0): {b_sinC.mean():.2f}')
axes[1].hist(b_conC, bins=30, alpha=0.55, color='darkorange',     label=f'Mal control: {b_conC.mean():.2f}')
axes[1].axvline(0, color='black', linestyle='--', lw=1.5)
axes[1].set_title('Caso 2: Colisionador\nCoef. de conexiones')
axes[1].set_xlabel('Coeficiente estimado')
axes[1].legend(fontsize=8)

# Caso 3: Proxy contaminado
axes[2].hist(b_sinP,  bins=30, alpha=0.55, color='mediumpurple', label=f'Sin control (~2): {b_sinP.mean():.2f}')
axes[2].hist(b_proxy, bins=30, alpha=0.55, color='tomato',       label=f'Proxy malo (<2): {b_proxy.mean():.2f}')
axes[2].hist(b_ideal, bins=30, alpha=0.55, color='steelblue',    label=f'Control ideal (~2): {b_ideal.mean():.2f}')
axes[2].axvline(2, color='black', linestyle='--', lw=1.5)
axes[2].set_title('Caso 3: Proxy contaminado\nCoef. de tratamiento')
axes[2].set_xlabel('Coeficiente estimado')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('bad_controls_montecarlo.png', dpi=120, bbox_inches='tight')
plt.show()
print("Gráfico guardado como bad_controls_montecarlo.png")

---
## Resumen: Los tres casos

| Caso | Estructura | Variable problemática | ¿Por qué parece buena idea? | Consecuencia de controlar |
|------|-----------|----------------------|----------------------------|---------------------------|
| **1. Mediador** | D → M → Y | M = tipo de empleo | M tiene efecto sobre Y | Bloquea el canal causal de D |
| **2. Colisionador** | D → C ← Y | C = ser contratado | C parece relevante para Y | Abre correlación espuria entre D e Y |
| **3. Proxy contaminado** | D → P ← U → Y | P = test tardío | P mide el confusor U | Absorbe parte del efecto de D |

### Checklist rápido antes de agregar un control
1. ¿Esta variable ocurre **antes** del tratamiento?
2. ¿Puede ser afectada por el tratamiento? (si sí → mediador o proxy contaminado)
3. ¿Es causada tanto por el tratamiento como por el resultado? (si sí → colisionador)
4. ¿Está en el camino causal entre D e Y? (si sí → no controlar si buscas efecto total)
5. Si la incluyo, ¿el coeficiente de D tiene el signo/magnitud que espero?
6. ¿Puedo trazar el DAG? ¿El control cierra una puerta trasera o abre un camino espurio?